# Colombo tea auction | Next-auction price forecasting

Forecast the next reported price midpoint in LKR for a recurring tea group. Compare five standalone ensemble models and a Random Forest + XGBoost stack against the last published price.

Dataset: [Mallawarachchi et al., Colombo Tea Auction data](https://huggingface.co/datasets/colombo-tea-auction-prices/cta-price-prediction). 

AI assistance: OpenAI Codex helped generate and verify the notebook code.

In [ ]:
from pathlib import Path
import hashlib
import math
import re

import numpy as np
import pandas as pd
from IPython.display import display

DATA_PATH = Path("colombo_tea_auction_dataset.csv")
TARGET = "price_mid_lkr"
TEA_FEATURES = ["catalogue", "category", "elevation", "segment", "grade", "tier", "region"]
if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Upload {DATA_PATH.name} to the current session directory.")

raw = pd.read_csv(DATA_PATH)
missing_prices = raw[TARGET].isna().sum()
data = raw.dropna(subset=[TARGET]).copy()
data[TARGET] = pd.to_numeric(data[TARGET], errors="raise")
assert np.isfinite(data[TARGET]).all()

def first_auction_day(label):
    # Dates are given as two auction days followed by the month and year.
    match = re.fullmatch(
        r"\s*(\d{1,2})(?:st|nd|rd|th)?(?:\s*/\s*\d{1,2}(?:st|nd|rd|th)?)?\s+([a-z]+)\s+(\d{4})\s*",
        str(label),
        flags=re.IGNORECASE,
    )
    if match is None:
        raise ValueError(f"Unexpected auction date: {label!r}")
    return pd.to_datetime(" ".join(match.groups()), format="%d %B %Y")

date_lookup = {label: first_auction_day(label) for label in data["sale_date_raw"].unique()}
data["auction_date"] = data["sale_date_raw"].map(date_lookup)
assert data.groupby("sale_id")["auction_date"].nunique().eq(1).all()

print(f"Dataset SHA-256: {hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()}")
print(f"{len(raw):,} source rows | {missing_prices:,} missing prices | {len(data):,} usable rows")
print(f"{data['sale_id'].nunique()} auctions across {data['auction_date'].nunique()} start dates")

Dataset SHA-256: 83c5da16ed07e4cf174d87ec6d625e67ab9286398ae6eab0d353a92e41a5b45f
12,233 source rows | 436 missing prices | 11,797 usable rows
105 auctions across 104 start dates


## Load the data and reserve the holdout

Use the first reported auction day for chronological ordering. Keep sales that share a reported date on the same side of the split. The reserved period starts on 28 October 2025.

In [2]:
auction_days = pd.DatetimeIndex(data["auction_date"].unique()).sort_values()
holdout_size = math.ceil(0.20 * len(auction_days))
test_start = auction_days[-holdout_size]
development = data.loc[data["auction_date"] < test_start].copy()
final_test = data.loc[data["auction_date"] >= test_start].copy()

assert development["auction_date"].max() < final_test["auction_date"].min()
assert set(development["sale_id"]).isdisjoint(final_test["sale_id"])


FINAL_TEST_START = pd.Timestamp("2025-10-28")
FINAL_TEST_SALE_IDS = frozenset(final_test["sale_id"])
EXPECTED_DATA_HASH = "83c5da16ed07e4cf174d87ec6d625e67ab9286398ae6eab0d353a92e41a5b45f"
assert hashlib.sha256(DATA_PATH.read_bytes()).hexdigest() == EXPECTED_DATA_HASH
assert test_start == FINAL_TEST_START and len(FINAL_TEST_SALE_IDS) == 21
assert len(final_test) == 2312
print(f"Reserved: {len(FINAL_TEST_SALE_IDS)} sales starting {FINAL_TEST_START.date()}")

Reserved: 21 sales starting 2025-10-28


## Next-auction forecast pairs

A forecast is made after the earlier auction's report is available. Match the same seven-field tea group from that report to the immediately following recorded auction. To keep a roughly weekly forecast horizon and avoid matching over likely missing sales, accept only gaps of 5 to 9 days and require consecutive sale numbers within a year. At a year boundary, require sale 1 in the next year. Use reported start dates as supplied. When more than one sale ID shares a date, skip links touching that date because their order is uncertain. Build the calendar before removing rows with missing prices so a missing quote never causes a jump to a later auction.

The nine model inputs exclude weather and include seven group fields, the last published midpoint, and the scheduled target auction month. The target is the following auction's reported midpoint. Split by target date, maintaining the existing 21-sale holdout. During development validation, training labels must be known by the first forecast origin in the validation block. A label from the origin auction itself is eligible once that report is available. The CSV records auction dates without report publication timestamps.

The holdout audit reports availability and coverage counts only. This section fits and scores no models.

In [3]:
from sklearn.model_selection import TimeSeriesSplit

PAIR_FEATURES = [*TEA_FEATURES, "last_price_mid_lkr", "next_auction_month"]
PAIR_TARGET = "next_price_mid_lkr"
assert len(PAIR_FEATURES) == 9 and PAIR_TARGET not in PAIR_FEATURES

reports = raw.copy()
report_dates = {label: first_auction_day(label) for label in reports["sale_date_raw"].unique()}
reports["auction_date"] = reports["sale_date_raw"].map(report_dates)
assert reports[TEA_FEATURES].notna().all().all()
assert not reports.duplicated(["sale_id", *TEA_FEATURES]).any()

sales = (
    reports[["sale_id", "auction_date", "sale_year", "sale_number"]]
    .drop_duplicates()
    .sort_values(["auction_date", "sale_number", "sale_id"])
    .reset_index(drop=True)
)
assert sales["sale_id"].is_unique
unambiguous = ~sales["auction_date"].duplicated(keep=False)
previous = sales.shift()
gap_days = (sales["auction_date"] - previous["auction_date"]).dt.days
same_year_successor = (
    sales["sale_year"].eq(previous["sale_year"])
    & sales["sale_number"].eq(previous["sale_number"] + 1)
)
new_year_successor = (
    sales["sale_year"].eq(previous["sale_year"] + 1)
    & sales["sale_number"].eq(1)
)
valid_link = (
    unambiguous & unambiguous.shift(fill_value=False)
    & gap_days.between(5, 9) & (same_year_successor | new_year_successor)
)
links = pd.DataFrame({
    "origin_sale_id": previous.loc[valid_link, "sale_id"],
    "origin_date": previous.loc[valid_link, "auction_date"],
    "target_sale_id": sales.loc[valid_link, "sale_id"],
    "target_date": sales.loc[valid_link, "auction_date"],
    "gap_days": gap_days.loc[valid_link].astype(int),
})

origin_quotes = reports[["sale_id", *TEA_FEATURES, TARGET]].rename(
    columns={"sale_id": "origin_sale_id", TARGET: "last_price_mid_lkr"}
)
target_quotes = reports[["sale_id", *TEA_FEATURES, TARGET]].rename(
    columns={"sale_id": "target_sale_id", TARGET: PAIR_TARGET}
)
target_rows = target_quotes.merge(links, on="target_sale_id", how="left", validate="many_to_one")
matched = target_rows.merge(
    origin_quotes, on=["origin_sale_id", *TEA_FEATURES], how="left", validate="many_to_one"
)
pairs = matched.dropna(subset=["last_price_mid_lkr", PAIR_TARGET]).copy()
pairs["next_auction_month"] = pairs["target_date"].dt.month
pairs = pairs.sort_values(["target_date", "target_sale_id", *TEA_FEATURES]).reset_index(drop=True)
assert pairs["origin_date"].lt(pairs["target_date"]).all()
assert not pairs.duplicated(["target_sale_id", *TEA_FEATURES]).any()

pair_test = pairs.loc[pairs["target_sale_id"].isin(FINAL_TEST_SALE_IDS)].copy()
pair_development = pairs.loc[pairs["target_date"] < FINAL_TEST_START].copy()
assert len(pair_test) + len(pair_development) == len(pairs)
assert pair_test["target_date"].ge(FINAL_TEST_START).all()
assert pair_development["target_date"].max() <= pair_test["origin_date"].min()

reserved = matched.loc[matched["target_sale_id"].isin(FINAL_TEST_SALE_IDS)]
has_target = reserved[PAIR_TARGET].notna()
has_previous = reserved["last_price_mid_lkr"].notna()
assert int((has_target & has_previous).sum()) == len(pair_test)
print(f"Valid weekly links: {len(links)} of {len(sales) - 1} adjacent recorded sale links")
print(f"Development: {len(pair_development):,} pairs across {pair_development['target_sale_id'].nunique()} target auctions")
print(f"Reserved: {len(reserved):,} rows across {len(FINAL_TEST_SALE_IDS)} target auctions")
print(f"Reserved rows with a usable previous price: {has_previous.sum():,}")
print(f"Reserved rows with both prices: {len(pair_test):,} of {has_target.sum():,} labelled rows")
print(f"Reserved target auctions with eligible pairs: {pair_test['target_sale_id'].nunique()}")

pair_validation_folds = []
pair_fold_summary = []
development_days = pd.DatetimeIndex(development["auction_date"].unique()).sort_values()
for fold, (_, validation_indices) in enumerate(TimeSeriesSplit(n_splits=4).split(development_days), start=1):
    block = pair_development.loc[
        pair_development["target_date"].isin(development_days[validation_indices])
    ]
    first_origin = block["origin_date"].min()
    train = pair_development.loc[
        (pair_development["target_date"] <= first_origin)
        & (pair_development["target_date"] < block["target_date"].min())
    ]
    assert not train.empty and not block.empty
    assert train["target_date"].max() <= first_origin
    assert set(train["target_sale_id"]).isdisjoint(block["target_sale_id"])
    pair_validation_folds.append((fold, train, block))
    pair_fold_summary.append({
        "fold": fold,
        "train pairs": len(train),
        "validation pairs": len(block),
        "last training target": train["target_date"].max().date(),
        "first validation origin": first_origin.date(),
        "first validation target": block["target_date"].min().date(),
        "last validation target": block["target_date"].max().date(),
    })
display(pd.DataFrame(pair_fold_summary))

Valid weekly links: 90 of 104 adjacent recorded sale links
Development: 7,592 pairs across 72 target auctions
Reserved: 2,433 rows across 21 target auctions
Reserved rows with a usable previous price: 1,802
Reserved rows with both prices: 1,764 of 2,312 labelled rows
Reserved target auctions with eligible pairs: 17
 fold  train pairs  validation pairs last training target first validation origin first validation target last validation target
    1         1580              1560           2024-03-19              2024-03-19              2024-03-26             2024-07-15
    2         3140              1721           2024-07-15              2024-07-15              2024-07-22             2024-11-05
    3         4861              1160           2024-11-05              2024-11-05              2024-11-11             2025-06-24
    4         6021              1571           2025-06-24              2025-06-24              2025-07-01             2025-10-21


## Define the models and evaluation metrics

The five standalone models use the same input columns and chronological folds. Random Forest, Extra Trees, XGBoost, and LightGBM encode the categorical columns within each training fold. CatBoost receives the categorical columns directly. The stack combines Random Forest and XGBoost through Ridge regression. MAE in LKR is the primary metric.

In [4]:
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder

print(f"Versions: numpy {np.__version__}, pandas {pd.__version__}, scikit-learn {sklearn.__version__}")

def make_forest(features, regressor=RandomForestRegressor):
    numeric = [column for column in features if column not in TEA_FEATURES]
    preprocessing = ColumnTransformer([
        ("tea", OneHotEncoder(handle_unknown="ignore"), TEA_FEATURES),
        ("numeric", SimpleImputer(strategy="median"), numeric),
    ])
    return make_pipeline(
        preprocessing,
        regressor(n_estimators=200, random_state=42, n_jobs=2),
    )

def metrics(actual, predicted):
    return {
        "MAE (LKR)": mean_absolute_error(actual, predicted),
        "RMSE (LKR)": np.sqrt(mean_squared_error(actual, predicted)),
        "R²": r2_score(actual, predicted),
    }

from sklearn.base import clone

import catboost
import lightgbm
import xgboost
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

print(f"XGBoost {xgboost.__version__}, CatBoost {catboost.__version__}, LightGBM {lightgbm.__version__}")

def make_boosting_models(features):
    numeric = [column for column in features if column not in TEA_FEATURES]
    encoded_features = ColumnTransformer([
        ("tea", OneHotEncoder(handle_unknown="ignore", sparse_output=False), TEA_FEATURES),
        ("numeric", SimpleImputer(strategy="median"), numeric),
    ]).set_output(transform="pandas")
    return [
        ("XGBoost", make_pipeline(encoded_features, XGBRegressor(
            n_estimators=300, learning_rate=0.05, max_depth=6,
            objective="reg:squarederror", tree_method="hist", random_state=42, n_jobs=2,
        ))),
        ("CatBoost", CatBoostRegressor(
            iterations=300, learning_rate=0.05, depth=6, cat_features=TEA_FEATURES,
            random_seed=42, thread_count=2, verbose=False, allow_writing_files=False,
        )),
        ("LightGBM", make_pipeline(clone(encoded_features), LGBMRegressor(
            n_estimators=300, learning_rate=0.05, num_leaves=31,
            random_state=42, n_jobs=2, verbosity=-1,
        ))),
    ]

from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

def make_stack_combiner():
    return make_pipeline(StandardScaler(), Ridge(alpha=1.0))

PAIR_BASELINE = "Last published price"
PAIR_STACK = "Random Forest + XGBoost stack"

def make_pair_base_models():
    features = list(PAIR_FEATURES)
    return {
        "Random Forest": make_forest(features),
        "Extra Trees": make_forest(features, regressor=ExtraTreesRegressor),
        **dict(make_boosting_models(features)),
    }

def pair_leaderboard(rows, model_columns):
    return pd.DataFrame([
        {"model": name, "validation rows": len(rows),
         **metrics(rows[PAIR_TARGET], rows[name])}
        for name in model_columns
    ]).sort_values("MAE (LKR)", ignore_index=True)


pair_stack_columns = ["Random Forest", "XGBoost"]
standalone_columns = [PAIR_BASELINE, *make_pair_base_models()]
comparison_columns = [*standalone_columns, PAIR_STACK]
PAIR_MODEL_NAMES = [*standalone_columns[1:], PAIR_STACK]

Versions: numpy 2.5.5, pandas 3.0.1, scikit-learn 1.7.2
XGBoost 3.4.1, CatBoost 1.2.10, LightGBM 4.7.0


## Validate price-change forecasting

Each model learns the change from the last published midpoint. The forecast adds the predicted change back to that midpoint. The stack learns from earlier out-of-fold predictions and is scored on folds 2 through 4. Its leaderboard compares all methods on the same rows.

In the development comparison, the six-model mean MAE was LKR 42.954 for price-change training and LKR 45.699 for direct-price training, on the same 4,452 rows. Price change was selected before final testing. The last published price remains the benchmark.

In [5]:
PAIR_LAST_PRICE = "last_price_mid_lkr"

change_oof_blocks = []
for fold, train, validation in pair_validation_folds:
    print(f"Training price-change fold {fold}", flush=True)
    train_change = train[PAIR_TARGET] - train[PAIR_LAST_PRICE]
    fold_output = validation[["origin_date", "target_date", PAIR_TARGET, PAIR_LAST_PRICE]].copy()
    fold_output["fold"] = fold
    fold_output[PAIR_BASELINE] = validation[PAIR_LAST_PRICE]
    for name, model in make_pair_base_models().items():
        model.fit(train[PAIR_FEATURES], train_change)
        fold_output[name] = validation[PAIR_LAST_PRICE] + model.predict(validation[PAIR_FEATURES])
        print(f"  {name} complete", flush=True)
    change_oof_blocks.append(fold_output)

change_oof = pd.concat(change_oof_blocks).sort_index()
assert change_oof.index.is_unique
change_stack_blocks = []
for fold in sorted(change_oof["fold"].unique())[1:]:
    current = change_oof.loc[change_oof["fold"].eq(fold)].copy()
    first_origin = current["origin_date"].min()
    earlier = change_oof.loc[
        change_oof["fold"].lt(fold) & change_oof["target_date"].le(first_origin)
    ]
    assert not earlier.empty and earlier["target_date"].max() <= first_origin
    combiner = make_stack_combiner()
    earlier_base_changes = earlier[pair_stack_columns].sub(earlier[PAIR_LAST_PRICE], axis="index")
    combiner.fit(earlier_base_changes, earlier[PAIR_TARGET] - earlier[PAIR_LAST_PRICE])
    current_base_changes = current[pair_stack_columns].sub(current[PAIR_LAST_PRICE], axis="index")
    current[PAIR_STACK] = current[PAIR_LAST_PRICE] + combiner.predict(current_base_changes)
    change_stack_blocks.append(current)

change_stack_validation = pd.concat(change_stack_blocks)
assert change_stack_validation.index.equals(change_oof.loc[change_oof["fold"].gt(1)].index)
display(pair_leaderboard(change_oof, standalone_columns).round(3))
display(pair_leaderboard(change_stack_validation, comparison_columns).round(3))

Training price-change fold 1
  Random Forest complete
  Extra Trees complete
  XGBoost complete
  CatBoost complete
  LightGBM complete
Training price-change fold 2
  Random Forest complete
  Extra Trees complete
  XGBoost complete
  CatBoost complete
  LightGBM complete
Training price-change fold 3
  Random Forest complete
  Extra Trees complete
  XGBoost complete
  CatBoost complete
  LightGBM complete
Training price-change fold 4
  Random Forest complete
  Extra Trees complete
  XGBoost complete
  CatBoost complete
  LightGBM complete
               model  validation rows  MAE (LKR)  RMSE (LKR)    R²
Last published price             6012     35.914      71.666 0.988
            CatBoost             6012     40.796      76.415 0.987
             XGBoost             6012     41.621      81.273 0.985
            LightGBM             6012     44.914      84.612 0.983
       Random Forest             6012     45.729      86.381 0.983
         Extra Trees             6012     48.178      

## Locked setup for the reserved test

The final comparison uses the nine features, the price-change target, fixed model settings, and the last-price benchmark. All methods are evaluated on the same eligible reserved rows using MAE, RMSE, and R-squared.

In [6]:
LOCKED_PAIR_FEATURES = tuple(PAIR_FEATURES)
LOCKED_PAIR_TARGET_FORMULATION = "price_change"
LOCKED_PAIR_PRIMARY_METRIC = "MAE (LKR)"
LOCKED_PAIR_MODELS = tuple(PAIR_MODEL_NAMES)

assert len(LOCKED_PAIR_FEATURES) == 9
assert LOCKED_PAIR_FEATURES == (*TEA_FEATURES, PAIR_LAST_PRICE, "next_auction_month")
assert len(pair_development) == 7592 and len(pair_test) == 1764
assert pair_test["target_sale_id"].nunique() == 17
print(f"Locked target: {LOCKED_PAIR_TARGET_FORMULATION}")
print(f"Development pairs: {len(pair_development):,}")
print(f"Reserved pairs: {len(pair_test):,} (unscored)")

Locked target: price_change
Development pairs: 7,592
Reserved pairs: 1,764 (unscored)


## Pre-test integrity checks

The holdout represents a rolling next-auction backtest with fixed model parameters. For a later holdout auction, an earlier holdout sale's published price can be used as the forecast input once its report would be available. The models and Ridge combiner are fitted on development data. The forecast origin is the availability of the previous report, with its sale date serving as a proxy because the CSV lacks exact release times.

The checks below validate split boundaries, source lineage, matched evaluation rows, and the inputs for the final Ridge fit. The audit does not generate holdout predictions or scores.

In [7]:
def audit_pair_partition(frame):
    assert frame.index.is_unique
    assert frame["target_date"].sub(frame["origin_date"]).dt.days.between(5, 9).all()
    assert frame["next_auction_month"].eq(frame["target_date"].dt.month).all()

    quote_key = ["origin_sale_id", *TEA_FEATURES]
    published = origin_quotes.set_index(quote_key)["last_price_mid_lkr"]
    origin_index = pd.MultiIndex.from_frame(frame[quote_key])
    np.testing.assert_array_equal(
        frame[PAIR_LAST_PRICE].to_numpy(), published.reindex(origin_index).to_numpy()
    )
    assert np.isfinite(frame[[PAIR_LAST_PRICE, "next_auction_month"]]).all().all()

for partition in (pair_development, pair_test):
    audit_pair_partition(partition)
assert set(pair_development["target_sale_id"]).isdisjoint(pair_test["target_sale_id"])
assert set(pair_test["target_sale_id"]).issubset(FINAL_TEST_SALE_IDS)
assert pair_development["target_date"].max() <= pair_test["origin_date"].min()
assert np.isfinite(pair_development[PAIR_TARGET]).all()
assert LOCKED_PAIR_TARGET_FORMULATION == "price_change"

all_validation_indices = pd.Index(np.concatenate([
    block.index.to_numpy() for _, _, block in pair_validation_folds
]))
assert all_validation_indices.is_unique
assert all_validation_indices.sort_values().equals(change_oof.index)
for fold, train, block in pair_validation_folds:
    assert train["target_date"].max() <= block["origin_date"].min()
    assert train.index.intersection(block.index).empty
    assert train.index.intersection(pair_test.index).empty
    assert block.index.intersection(pair_test.index).empty
assert change_oof[PAIR_TARGET].equals(pair_development.loc[change_oof.index, PAIR_TARGET])
assert np.isfinite(change_oof[pair_stack_columns]).all().all()

# The final combiner takes changes, in the same column order as in validation.
final_combiner_inputs = change_oof[pair_stack_columns].sub(change_oof[PAIR_LAST_PRICE], axis="index")
final_combiner_target = change_oof[PAIR_TARGET] - change_oof[PAIR_LAST_PRICE]
assert final_combiner_inputs.index.equals(final_combiner_target.index)

skipped_sales = sorted(FINAL_TEST_SALE_IDS - set(pair_test["target_sale_id"]))
print("Verified holdout boundaries, last-price lineage, and validation row alignment.")
print(f"Final Ridge training inputs: {len(final_combiner_inputs):,} development predictions")
print("Reserved sales with no eligible pairs:", ", ".join(skipped_sales))
print("Reserved pairs remain unscored.")

Verified holdout boundaries, last-price lineage, and validation row alignment.
Final Ridge training inputs: 6,012 development predictions
Reserved sales with no eligible pairs: SALE_2025_47, SALE_2026_01, SALE_2026_10, SALE_2026_14
Reserved pairs remain unscored.


## Final reserved-auction evaluation

The five standalone models are refit on all development pairs. The Ridge combiner is fitted on development out-of-fold predictions. These fits are fixed throughout the holdout, and rankings use MAE in LKR.

In [8]:
assert LOCKED_PAIR_TARGET_FORMULATION == "price_change"
features = list(LOCKED_PAIR_FEATURES)
development_change = pair_development[PAIR_TARGET] - pair_development[PAIR_LAST_PRICE]

final_base_models = make_pair_base_models()
for name, model in final_base_models.items():
    model.fit(pair_development[features], development_change)
    print(f"Fitted {name} on {len(pair_development):,} development pairs", flush=True)

final_ridge = make_stack_combiner()
final_ridge.fit(final_combiner_inputs, final_combiner_target)

holdout_features = pair_test[features]
holdout_changes = pd.DataFrame({
    name: model.predict(holdout_features)
    for name, model in final_base_models.items()
}, index=pair_test.index)
holdout_changes[PAIR_STACK] = final_ridge.predict(holdout_changes[pair_stack_columns])

final_predictions = pair_test[["target_sale_id", "target_date", *TEA_FEATURES, PAIR_TARGET]].copy()
last_price = holdout_features[PAIR_LAST_PRICE]
final_predictions[PAIR_BASELINE] = last_price
for name in LOCKED_PAIR_MODELS:
    final_predictions[name] = last_price + holdout_changes[name]

scored_methods = [PAIR_BASELINE, *LOCKED_PAIR_MODELS]
assert final_predictions.index.equals(pair_test.index)
assert np.isfinite(final_predictions[[PAIR_TARGET, *scored_methods]]).all().all()

final_scores = pd.DataFrame([
    {"method": name, "test rows": len(final_predictions),
     **metrics(final_predictions[PAIR_TARGET], final_predictions[name])}
    for name in scored_methods
]).sort_values(LOCKED_PAIR_PRIMARY_METRIC, ignore_index=True)
display(final_scores.round(3))
print(f"Tested {len(final_predictions):,} pairs across {final_predictions['target_sale_id'].nunique()} reserved auctions.")
print(f"Lowest test MAE: {final_scores.loc[0, 'method']}")

Fitted Random Forest on 7,592 development pairs
Fitted Extra Trees on 7,592 development pairs
Fitted XGBoost on 7,592 development pairs
Fitted CatBoost on 7,592 development pairs
Fitted LightGBM on 7,592 development pairs
                       method  test rows  MAE (LKR)  RMSE (LKR)    R²
         Last published price       1764     36.054      68.264 0.988
Random Forest + XGBoost stack       1764     36.736      68.611 0.988
                     CatBoost       1764     38.878      70.865 0.987
                     LightGBM       1764     41.623      79.221 0.984
                      XGBoost       1764     42.694      78.398 0.984
                Random Forest       1764     44.598      77.951 0.984
                  Extra Trees       1764     48.876      86.849 0.980
Tested 1,764 pairs across 17 reserved auctions.
Lowest test MAE: Last published price


## Export assets for the dashboard

The export bundles the fitted models and preprocessing with the tea group catalogue, holdout predictions, and test metrics. The dashboard loads these files for forecasts and model comparison.

In [9]:
import json
import joblib

dashboard_assets = Path("dashboard_assets")
dashboard_assets.mkdir(exist_ok=True)
bundle = {
    "features": features,
    "tea_features": TEA_FEATURES,
    "models": final_base_models,
    "combiner": final_ridge,
    "stack_inputs": pair_stack_columns,
    "stack_name": PAIR_STACK,
    "baseline_name": PAIR_BASELINE,
}
model_path = dashboard_assets / "models.joblib"
joblib.dump(bundle, model_path, compress=3)

# Confirm that the serialized estimators preserve the tested predictions.
restored = joblib.load(model_path)
for name, model in restored["models"].items():
    np.testing.assert_allclose(model.predict(holdout_features), holdout_changes[name])
np.testing.assert_allclose(
    restored["combiner"].predict(holdout_changes[restored["stack_inputs"]]),
    holdout_changes[PAIR_STACK],
)

pair_development[TEA_FEATURES].drop_duplicates().sort_values(TEA_FEATURES).to_csv(
    dashboard_assets / "tea_groups.csv", index=False
)
final_predictions.to_csv(dashboard_assets / "test_predictions.csv", index=False)
final_scores.to_csv(dashboard_assets / "test_scores.csv", index=False)
metadata = {
    "target": PAIR_TARGET,
    "development_pairs": len(pair_development),
    "test_pairs": len(final_predictions),
    "test_auctions": final_predictions["target_sale_id"].nunique(),
    "test_start": final_predictions["target_date"].min().date().isoformat(),
    "test_end": final_predictions["target_date"].max().date().isoformat(),
    "benchmark": PAIR_BASELINE,
    "best_model": final_scores.loc[final_scores["method"].ne(PAIR_BASELINE), "method"].iloc[0],
}
(dashboard_assets / "metadata.json").write_text(json.dumps(metadata, indent=2) + "\n", encoding="utf-8")
print(f"Exported and verified dashboard assets ({model_path.stat().st_size / 1_000_000:.1f} MB model bundle).")

Exported and verified dashboard assets (50.8 MB model bundle).
